# Part B – Dataset Understanding & Preparation

This notebook solves:

1. Identify input features and target variable
2. Perform train-test split while maintaining class distribution
3. Identify missing values and apply KNN Imputer for multivariate imputation


In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('Risk_Alert_Classifier_Dataset_4600 - Risk_Alert_Classifier_Dataset_4600.csv.csv')

print("Shape:", df.shape)
df.head()


Shape: (4600, 19)


,customer_id,age,gender,region,employment_type,annual_income_inr,credit_score,credit_utilization_ratio,missed_payments_12m,avg_late_payment_days,monthly_transaction_count,monthly_spend_inr,cash_advance_count_6m,complaints_last_6m,failed_login_attempts_3m,account_tenure_months,last_transaction_date,debt_balance_inr,risk_status
0,500001,43.0,Female,NaN,Salaried,82242.0,NaN,0.120,1,2.2,39,33889.0,0,2,4,70,2025-09-26,87273,0
1,500002,29.0,Female,Central,Salaried,32769.0,647.0,0.337,1,1.5,11,10853.0,1,1,1,34,2025-11-24,20600,0
2,500003,36.0,Male,East,Salaried,39731.0,727.0,0.175,0,3.9,45,25519.0,2,1,1,74,2025-09-26,47565,0
3,500004,28.0,Male,North,Unemployed,38990.0,553.0,0.472,7,23.3,103,17806.0,1,2,6,72,2025-10-03,43803,1
4,500005,36.0,Female,East,Self-Employed,41043.0,732.0,0.418,1,9.8,95,27114.0,0,1,1,11,2025-10-26,12008,0


## Task 7 – Identify Input Features and Target Variable

In [2]:
target = 'risk_status'

X = df.drop(columns=[target])
y = df[target]

print("Target Variable:", target)
print("\nInput Features:")
print(list(X.columns))


Target Variable: risk_status

Input Features:
['customer_id', 'age', 'gender', 'region', 'employment_type', 'annual_income_inr', 'credit_score', 'credit_utilization_ratio', 'missed_payments_12m', 'avg_late_payment_days', 'monthly_transaction_count', 'monthly_spend_inr', 'cash_advance_count_6m', 'complaints_last_6m', 'failed_login_attempts_3m', 'account_tenure_months', 'last_transaction_date', 'debt_balance_inr']


In [3]:
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer_id                4600 non-null   int64  
 1   age                        4460 non-null   float64
 2   gender                     4600 non-null   object 
 3   region                     4498 non-null   object 
 4   employment_type            4456 non-null   object 
 5   annual_income_inr          4434 non-null   float64
 6   credit_score               4384 non-null   float64
 7   credit_utilization_ratio   4453 non-null   float64
 8   missed_payments_12m        4600 non-null   int64  
 9   avg_late_payment_days      4600 non-null   float64
 10  monthly_transaction_count  4600 non-null   int64  
 11  monthly_spend_inr          4471 non-null   float64
 12  cash_advance_count_6m      4600 non-null   int64  
 13  complaints_last_6m         4600 non-null   int64

## Missing Value Analysis

In [4]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print(missing)


credit_score                216
annual_income_inr           166
credit_utilization_ratio    147
employment_type             144
age                         140
monthly_spend_inr           129
region                      102
dtype: int64


## Convert Date Column

In [5]:
df['last_transaction_date'] = pd.to_datetime(df['last_transaction_date'])

df['last_transaction_year'] = df['last_transaction_date'].dt.year
df['last_transaction_month'] = df['last_transaction_date'].dt.month
df['last_transaction_day'] = df['last_transaction_date'].dt.day

df.drop('last_transaction_date', axis=1, inplace=True)


## Encode Categorical Features

In [6]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()

categorical_cols = df_encoded.select_dtypes(include='object').columns

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = df_encoded[col].astype(str)
    df_encoded[col] = le.fit_transform(df_encoded[col])
    encoders[col] = le

df_encoded.head()


,customer_id,age,gender,region,employment_type,annual_income_inr,credit_score,credit_utilization_ratio,missed_payments_12m,avg_late_payment_days,...,monthly_spend_inr,cash_advance_count_6m,complaints_last_6m,failed_login_attempts_3m,account_tenure_months,debt_balance_inr,risk_status,last_transaction_year,last_transaction_month,last_transaction_day
0,500001,43.0,0,5,1,82242.0,NaN,0.120,1,2.2,...,33889.0,0,2,4,70,87273,0,2025,9,26
1,500002,29.0,0,0,1,32769.0,647.0,0.337,1,1.5,...,10853.0,1,1,1,34,20600,0,2025,11,24
2,500003,36.0,1,1,1,39731.0,727.0,0.175,0,3.9,...,25519.0,2,1,1,74,47565,0,2025,9,26
3,500004,28.0,1,2,4,38990.0,553.0,0.472,7,23.3,...,17806.0,1,2,6,72,43803,1,2025,10,3
4,500005,36.0,0,1,2,41043.0,732.0,0.418,1,9.8,...,27114.0,0,1,1,11,12008,0,2025,10,26


## Task 8 – Train Test Split with Stratification

In [7]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('risk_status', axis=1)
y = df_encoded['risk_status']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

print("\nTrain Class Distribution")
print(y_train.value_counts(normalize=True))

print("\nTest Class Distribution")
print(y_test.value_counts(normalize=True))


Train Shape: (3680, 20)
Test Shape: (920, 20)

Train Class Distribution
risk_status
0    0.878804
1    0.121196
Name: proportion, dtype: float64

Test Class Distribution
risk_status
0    0.879348
1    0.120652
Name: proportion, dtype: float64


## Task 9 – Apply KNN Imputer

In [8]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns
)

print("Missing Values in Training Set After Imputation")
print(X_train_imputed.isnull().sum().sum())

print("\nMissing Values in Testing Set After Imputation")
print(X_test_imputed.isnull().sum().sum())


Missing Values in Training Set After Imputation
0

Missing Values in Testing Set After Imputation
0


## Interpretation

### Input Features
All columns except `risk_status` are input features.

### Target Variable
`risk_status`
- 0 = Low Risk
- 1 = High Risk

### Train-Test Split
Stratified splitting preserves the original class distribution in both training and testing datasets.

### KNN Imputation
KNN Imputer replaces missing values using the nearest observations, preserving multivariate relationships better than mean/median imputation.
